# 🛡️ Multimodal Toxicity Detection Platform - Colab Deployment
This notebook is pre-configured to run the **HG Multimodal** platform (supporting English, Hindi, and Offline Media).

### Instructions:
1. Click **Runtime** -> **Run All**.
2. Wait for the **Vite Proxy URL** to appear at the bottom.
3. **ENSURE GPU IS ENABLED**: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU.

In [ ]:
# 1. Setup Environment and Dependencies
import os
print("Installing system dependencies (FFmpeg)...")
!sudo apt install ffmpeg -y

print("Installing Python libraries (Whisper, XLM-RoBERTa, yt-dlp)...")
!pip install -r requirements.txt
!pip install sacremoses openai-whisper moviepy python-multipart yt-dlp

print("Installing UI dependencies... (this may take 5-7 minutes, stay patient)")
!cd ui && npm install --quiet

print("✅ Setup Complete!")

In [ ]:
# 2. Launch Servers in Background
import threading
import time
import subprocess
import os
import uvicorn
import nest_asyncio
from src.api import app
from google.colab.output import eval_js

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
nest_asyncio.apply()

def run_backend():
    print("🚀 Starting Backend API (Port 8000)...")
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

def run_frontend():
    print("🎨 Starting Frontend UI (Port 5173)...")
    !cd ui && npm run dev -- --host 127.0.0.1 --port 5173

# Pre-cleanup
!fuser -k 8000/tcp
!fuser -k 5173/tcp

# Start Backend
t1 = threading.Thread(target=run_backend, daemon=True)
t1.start()

time.sleep(10) # Give API time to load

# Start Frontend
t2 = threading.Thread(target=run_frontend, daemon=True)
t2.start()

proxy_url = eval_js("google.colab.kernel.proxyPort(5173)")
print(f"\n--- ALL SERVERS RUNNING ---\n")
print(f"✅ OPEN APP HERE: {proxy_url}\n")